# 17. 預報檢驗 I：似然與一致性檢驗

前面七章一直在造模型：{doc}`第 10 章 <10_point_process>`給了共同語言，
第 11 到 14 章把目錄統計與 ETAS 拆到零件層級，第 15、16 章走到中期
前兆模型。每一章結尾都留著同一個沒回答的問題：**這些模型，哪一個
該被相信？** 這一章與下一章要把這個問題變成可以計算的東西。

我們手上其實已經有答案的原料。{eq}`eq:pp-loglik` 說，一個條件強度
為 $\lambda^*$ 的模型賦予一份目錄的對數概似是

$$\ln L = \sum_i \ln\lambda^*(t_i,x_i,y_i,m_i)
  - \int\!\!\int\!\!\int \lambda^*\,\mathrm{d}m\,\mathrm{d}x\,\mathrm{d}y\,\mathrm{d}t$$

押中有賞、亂押有罰。這條式子是**擬合**時被最佳化的目標函數；本章要做
的是把同一條式子挪到另一個位置——不再拿它調參數，而是拿它當**裁判**：
參數已經在觀測之前釘死了，現在只問模型對「真的發生的那些地震」給了
多高的分數，以及這個分數跟模型自己所預期的分數比起來算不算異常。

這個位置的挪動帶來三個必須從頭處理的技術問題。第一，國際上流通的
預報幾乎都不是連續的 $\lambda^*$，而是**網格化**的期望數
$\Lambda_{jk}$，所以要先弄清楚離散化到底做了什麼（17.2、17.3）。
第二，一個分數本身不是結論，要有**對照分布**才知道它算高算低，而
這個分布幾乎沒有解析解，只能模擬（17.5）。第三，Poisson 假設對真實
地震活動來說太緊了，得放寬（17.6、17.7）。

本章只處理「模型 vs. 資料」的一致性檢驗。「模型 vs. 模型」的比較檢驗、
資訊增益、警報式評估與統計功效，全部留給
{doc}`第 18 章 <18_testing_comparison>`。

## 17.1 為何要制度化

先講一段不太光彩的歷史。地震預測研究在 1990 年代前後跌到聲譽谷底，
Geller（1997）那篇〈Earthquake prediction: a critical review〉幾乎是
對整個次領域的判決書。但真正的病灶並不是「想法不夠聰明」。回頭看那些
爭議個案，共同結構驚人地一致：模型參數是在**看過**目標地震之後才定
下來的（或至少沒有辦法證明不是）；「預報」的空間範圍、時間窗、規模
門檻在事後被重新描述，命中與否變成一件可以辯論的事；用來判定的目錄
版本沒有指定，而目錄會被修訂；別人拿不到程式碼與資料，重跑不出同一
張圖。前兩項是**可否證性**的問題，後兩項是**可重現性**的問題，而兩者
都不是靠同儕審查能解決的：審查者看到的是一篇已經寫完的論文，裡面每個
選擇看起來都很合理，因為不合理的選擇早在寫作前就被作者自己刪掉了。

CSEP（Collaboratory for the Study of Earthquake Predictability）的解方
是把科學程序**制度化**，核心設計只有一句話：預報必須是**零自由度
（zero-degree-of-freedom）的可否證陳述**。在觀測窗開始之前，下面每一
項都要被第三方測試中心存檔釘死：

1. **模型參數**：不是「模型形式」，是每一個數值；校準用的目錄截止
   時間也要寫明。
2. **預報格式**：測試區邊界、空間格大小、規模箱寬、時間窗、深度
   範圍、每箱的期望數。
3. **目標資料來源**：哪個機構的哪個目錄、哪個規模尺度、哪個版本、
   目標地震門檻 $m_T$ 與挑選規則。
4. **評估方法**：跑哪些檢驗、顯著水準多少、多重檢定怎麼校正。

四項釘死之後，模型作者在觀測期間**能做的事情是零**，這就是「零自由度」
的意思。它的價值不在統計上多高明，而在於把「我保證我沒有偷看資料」
這種只能靠人格背書的宣稱，換成一個時間戳記。**科學程序不該依賴自律**，
這是 CSEP 最重要的一句話，也是本章所有技術細節存在的理由。制度化還
逼出了**共同格式**：要讓第三方自動評分，所有模型就得交出同一種資料
結構，而這個格式反過來決定了可用的統計方法——下一節就從這裡開始。

## 17.2 預報的兩種表示

目前流通的機率式地震預報只有兩種表示法，而**表示形式決定了可用的
檢驗**。這句話值得先記住，因為本章後面九成的內容都只適用於第一種。

### 表示一：網格化預報（grid-based）

把測試區切成空間格 $j=1,\dots,l$（CSEP 慣例是 $0.1^\circ\times0.1^\circ$）
與規模箱 $k=1,\dots,m$（慣例是 0.1 規模單位），模型對每個 $(j,k)$ 交出
預報期間的**期望地震數** $\Lambda_{jk}$。整份預報就是一個 $l\times m$
的非負矩陣，觀測就是同形狀的計數矩陣 $\omega_{jk}$。這個矩陣跟
{doc}`第 10 章 <10_point_process>`的連續率密度的關係是**積分**：

$$\Lambda_{jk} = \mathbb{E}\left[\int_{T_0}^{T_1}\!\!\int_{S_j}\!
  \int_{m_k}^{m_k+\Delta m}
  \lambda^*(t,x,y,m)\,\mathrm{d}m\,\mathrm{d}x\,\mathrm{d}y\,\mathrm{d}t\right]$$

兩個細節不能略過。**其一，這是四重積分**，網格化同時抹掉了箱內的
時間、空間與規模結構——一整年份的餘震序列被壓成一個數字，先來後到
不再有分別。**其二，外面那個期望值符號不是裝飾**。$\lambda^*$ 條件在
歷史 $H_t$ 上，而預報期間的歷史在發預報的當下還沒發生，所以
$\Lambda_{jk}$ 是「對所有可能的未來歷史取期望」之後的量。對時間獨立
模型這個期望是空的；對 ETAS 這種自激模型，它是一個真正的多重期望，
實務上只能用模擬去逼近。**這正是第二種表示法存在的理由。**

### 表示二：目錄式預報（catalog-based）

不交矩陣，交**一大堆模擬目錄**——例如把 ETAS 或 UCERF3-ETAS 跑一萬次，
每次得到一份完整的合成目錄（時間、位置、規模一應俱全），這一萬份目錄
的**經驗分布**就是預報本身。兩者的差別遠比「格式不同」深刻：

| 面向 | 網格化 | 目錄式 |
|---|---|---|
| 預報物件 | 每箱期望數 $\Lambda_{jk}$ | 目錄的經驗分布 |
| 箱內分布 | 需另外假設（通常 Poisson） | 由模擬自然給出 |
| 箱間相關 | 通常假設獨立 | 模擬中自然保留 |
| 可用檢驗 | Poisson 家族（本章） | 經驗分位數（17.10） |

網格化預報把「期望數」交出來之後，**箱內事件數的分布是未知的**——模型
只說了平均值，要算概似就必須補一個分布假設，而最方便的補法就是
Poisson。目錄式預報沒有這個問題：一萬份模擬目錄裡，某格出現 0、1、2、
5 顆的相對次數就是分布，叢集造成的過度離散與箱間相關也都原樣保留。
代價是計算量、儲存量與一整套要重新設計的統計方法，所以現實是：**絕大
多數已發表的前瞻檢驗結果，用的是網格化表示與 Poisson 假設**。這一章把
那一套講到底，同時把每一個假設標記清楚，這樣 17.6、17.7、17.10 的放寬
版本才知道自己在放寬什麼。

## 17.3 Poisson 似然

補上的假設有兩條：**各箱獨立**，且箱 $(j,k)$ 內的事件數服從期望為
$\Lambda_{jk}$ 的 Poisson 分布。於是單箱觀測到 $\omega$ 個事件的機率是

$$P(\omega\mid\Lambda) = \frac{\Lambda^{\omega}}{\omega!}\,e^{-\Lambda}$$

取對數，得到 CSEP 文獻裡稱為 **POLL**（Poisson log-likelihood）的
單箱分數：

$$\begin{aligned}
\mathrm{POLL}(\omega,\Lambda)
  &= \ln\!\left(\frac{\Lambda^{\omega}}{\omega!}e^{-\Lambda}\right) \\
  &= \omega\ln\Lambda - \ln\omega! - \Lambda \\
  &= -\Lambda + \omega\ln\Lambda - \ln\omega!
\end{aligned}$$

三項各有身分。$-\Lambda$ 是**押注成本**：模型在這格喊多高的率就先付
多少代價。$\omega\ln\Lambda$ 是**押中的回報**：事件真的發生時模型喊得
愈高賺得愈多（$\Lambda<1$ 時 $\ln\Lambda<0$，所以這一項通常是負的，
它獎勵的是「相對而言喊得高」）。$-\ln\omega!$ 是**組合修正項**，補償
箱內 $\omega$ 個事件的排列順序不可分辨。

各箱獨立，所以聯合對數概似就是逐箱相加，稱為 **jPOLL**：

$$\mathrm{jPOLL} = \sum_{j=1}^{l}\sum_{k=1}^{m}
  \Bigl[-\Lambda_{jk} + \omega_{jk}\ln\Lambda_{jk}
  - \ln\bigl(\omega_{jk}!\bigr)\Bigr]$$ (eq:jpoll)

### $\ln\omega!$ 到底重不重要

常聽到的說法是「$\ln\omega!$ 是常數，可以丟掉」。這句話對一半，而錯的
那一半會讓人在寫模擬程式時出錯。$\ln\omega_{jk}!$ **只跟觀測有關、完全
不含模型的量**，因此比較兩個模型 $A$ 與 $B$ 時，

$$\mathrm{jPOLL}_A - \mathrm{jPOLL}_B
  = \sum_{j,k}\Bigl[(\Lambda^B_{jk}-\Lambda^A_{jk})
  + \omega_{jk}\ln\frac{\Lambda^A_{jk}}{\Lambda^B_{jk}}\Bigr]$$

階乘項整項消掉，**模型比較（第 18 章）確實可以無視它**。但一致性檢驗
是拿觀測分數去比**模擬分數的分布**，而模擬目錄的 $\omega_{jk}$ 每次
都不一樣，階乘項因此**不是常數**——它會改變模擬分布的形狀。正確的
說法是：**兩邊都要算，而且要用同一條式子算**；只省一邊就是 bug。

### 與連續版概似的對應

網格化分數與 {eq}`eq:pp-loglik` 的連續分數是什麼關係？把箱體積寫成
$\Delta V = \Delta t\,\Delta A\,\Delta m$；箱細到每箱最多一顆事件時
$\ln\omega_{jk}!=0$、$\Lambda_{jk}\approx\lambda^*\Delta V$，代入
{eq}`eq:jpoll`：

$$\mathrm{jPOLL} \;\longrightarrow\;
  \sum_{i=1}^{N}\ln\lambda^*(t_i,x_i,y_i,m_i)
  \;-\;\int\lambda^*\;+\;N\ln\Delta V$$

也就是**連續對數概似加上一個只跟解析度有關的常數** $N\ln\Delta V$
（完整推導見 17.11 附錄 A）。這個對應有三個立即的後果：

- **jPOLL 的絕對值沒有意義**。換一個格子大小就整個平移。看到「這個
  模型的對數概似是 $-412.7$」時，唯一該問的是「跟誰比」。
- **跨測試區、跨解析度比分數是無效的**。加州 7682 格與義大利 8993 格
  的 jPOLL 不能並排；Bayona et al.（2023）為此改報「每顆地震的平均
  分數」，正是為了消掉這個尺度。
- **網格化是有損壓縮**，$\Delta V\to0$ 時分數才回到連續版；但格子愈粗
  檢驗力未必愈差，這個違反直覺的反轉留到第 18 章談功效時處理。

## 17.4 一致性檢驗四件套

有了 jPOLL，就可以問「模型與資料合不合」。但直接拿 {eq}`eq:jpoll`
去問，會得到一個很沒用的答案：分數低，然後呢？是率抓錯了？空間圖形
畫歪了？還是 $b$ 值不對？三種病一個症狀，沒有診斷價值。CSEP 的設計
哲學因此是**拆開來，一次只檢查一個面向**——四個一致性檢驗的差別，
全在於它們**刻意抽掉了什麼資訊**：

| 檢驗 | 檢什麼 | 刻意抽掉什麼 | 典型失敗訊息 |
|---|---|---|---|
| N-test | 總地震率 | 空間、規模 | 高估或低估活動度 |
| M-test | 規模分布 | 空間、總數 | $b$ 值或截止規模不對 |
| S-test | 空間圖形 | 規模、總數 | 震央落在模型沒預期的地方 |
| cL-test | 空間 × 規模聯合 | 總數（條件在 $N_{\rm obs}$） | 聯合結構不合 |

### N-test：只留總數

最單純的一個。模型的預報總數是 $N_{\rm fore}=\sum_{j,k}\Lambda_{jk}$、
觀測總數是 $N_{\rm obs}=\sum_{j,k}\omega_{jk}$。獨立 Poisson 的和仍是
Poisson（率相加），所以總數的分布有解析解
$N\sim\mathrm{Poisson}(N_{\rm fore})$，不必模擬；問的是 $N_{\rm obs}$
有沒有落在中央區間內。因為高估與低估都是有意義的失敗，N-test 是
**雙尾**的，通常報兩個分位數分數 $\delta_1 = P(N\ge N_{\rm obs})$ 與
$\delta_2 = P(N\le N_{\rm obs})$，任一個小於 $\alpha/2$ 就判定不一致。

In [ ]:
from gdms_toolkit.viz import setup_plotly
setup_plotly()

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from scipy.special import gammaln

from gdms_toolkit.viz import (ACCENT, PALETTE, QUAKE_COLOR, SEQUENTIAL,
                              apply_layout)

N_fore = 20.0                                   # 模型的預報總數
kk = np.arange(0, 46)
pmf_pois = stats.poisson.pmf(kk, N_fore)
lo, hi = stats.poisson.ppf([0.025, 0.975], N_fore)

obs_a, obs_b = 24, 38                           # 兩個假想的觀測總數
d1_a = 1.0 - stats.poisson.cdf(obs_a - 1, N_fore)
d1_b = 1.0 - stats.poisson.cdf(obs_b - 1, N_fore)

fig = go.Figure(go.Bar(x=kk, y=pmf_pois, marker_color=ACCENT, opacity=0.75,
                       name=f"Poisson({N_fore:.0f}) 預報數分布"))
fig.add_vrect(x0=lo - 0.5, x1=hi + 0.5, fillcolor=PALETTE[2], opacity=0.10,
              line_width=0,
              annotation_text=f"95% 區間 [{lo:.0f}, {hi:.0f}]")
fig.add_vline(x=obs_a, line_color=PALETTE[2], line_width=3,
              annotation_text=f"觀測 A={obs_a}（δ₁={d1_a:.2f}，一致）")
fig.add_vline(x=obs_b, line_color=QUAKE_COLOR, line_width=3,
              annotation_text=f"觀測 B={obs_b}（δ₁={d1_b:.1e}，不一致）")
apply_layout(fig, title="N-test：觀測總數落在模型預報數分布的哪裡",
             xaxis_title="預報期間的目標地震數",
             yaxis_title="機率", hovermode="x")
fig

模型說期望 20 顆。觀測 A 的 24 顆落在 95% 區間裡，**與模型一致**；
觀測 B 的 38 顆落在極右尾，$\delta_1$ 小到 $10^{-4}$ 量級，判定模型
**顯著低估**了地震率。判讀規則簡單到可以背：綠色在區間內、紅色在
區間外。但先埋一個伏筆：Poisson 的變異數等於平均，95% 區間只有大約
$\pm9$ 顆寬，而真實地震活動（尤其是沒有除叢的目錄）的年際變異遠大於
此。**觀測 B 到底是模型錯了，還是這個分布太窄？** 17.6 節回來處理。

### S-test：正規化掉率，只留形狀

空間檢驗的邏輯要繞一個彎。先把規模維度加總掉，
$\Lambda_{j\cdot}=\sum_k \Lambda_{jk}$，得到純空間的率場。此時直接算
jPOLL 會出事：**率抓錯的模型會在空間檢驗裡被判死**，因為 $-\Lambda$
與 $\omega\ln\Lambda$ 都含有總量的資訊——那就跟 N-test 重複了，而且
分不清失敗來自哪裡。解法是**把率場整體縮放到觀測總數**：

$$\tilde\Lambda_{j} = \Lambda_{j\cdot}\cdot
  \frac{N_{\rm obs}}{\sum_{j'}\Lambda_{j'\cdot}}$$

縮放之後 $\sum_j\tilde\Lambda_j = N_{\rm obs}$，總數的資訊被**刻意
抽乾**，剩下的純粹是「率在空間上的相對形狀」；S-test 就是拿
$\tilde\Lambda_j$ 算 jPOLL，再跟同樣正規化條件下的模擬分布比。
這一步既是整套設計最精巧的地方，也是最容易被誤解的地方。**S-test 對
「模型率是不是抓對」完全不敏感**——一個把全台灣地震率乘以 100 的
模型，S-test 分數一點都不會變。這是設計，不是缺陷。它也是**單尾**的，
只看下尾：分數太低表示地震落在模型認為不可能的地方，是失敗；分數
太高只表示地震剛好都落在模型的高率格，不構成「不一致」。

### M-test：對稱的另一半

把空間加總掉，$\Lambda_{\cdot k}=\sum_j \Lambda_{jk}$，同樣正規化到
$N_{\rm obs}$，檢查規模分布，結構與 S-test 完全對稱。

實務上 M-test 常常最沒有鑑別力，理由很現實：**參賽模型幾乎都用
Gutenberg–Richter 分配規模**，而且 $b$ 值往往取自同一份區域目錄，
大家的規模維度長得一模一樣——Bayona et al.（2022）的加州實驗因此直接
不報 M-test。這不代表它沒用（模型自帶特徵地震假設或用了不同 $b$ 值時
很關鍵），只是提醒：**檢驗的鑑別力取決於參賽者之間有沒有差異。**

### cL-test：條件在觀測數上的聯合檢驗

S 與 M 各看一個邊際分布，但**邊際都對不代表聯合對**：模型可能空間
圖形對、規模分布也對，卻把大地震擺在錯的地方。cL-test 檢查的正是
這個聯合結構，用完整的 $l\times m$ 矩陣算 {eq}`eq:jpoll`。它與 S、M
的差別在正規化方式：cL-test **不縮放率場**，而是在**模擬時固定事件
數為 $N_{\rm obs}$**——這就是「條件在觀測數上」的意思，用條件分布把
總數的自由度拿掉，但計分仍用模型交出來的原始 $\Lambda_{jk}$。

### L-test 為什麼被冷落

最原始的檢驗其實是 **L-test**：不做任何正規化與條件化，直接拿
{eq}`eq:jpoll` 的觀測值去比「完全按模型模擬」（連事件數都由模型的
Poisson 決定）的分布。它在數學上最乾淨，卻有一個致命傷：

**L-test 對總數過度敏感。** $-\sum\Lambda_{jk}$ 與 $\omega\ln\Lambda$
的量級都由總數主導，所以一個率高估兩倍、但空間圖形完美的模型會被
直接拒絕；而拒絕的訊息與「空間圖形完全畫錯、率剛好對」的模型長得
一模一樣。**兩種失敗模式被混成同一個紅燈**，診斷價值歸零。

cL-test 就是為了拆開這個混淆而生：把「數量對不對」丟給 N-test，把
「形狀對不對」留給 cL-test，各自給一個獨立的燈號。這也是為什麼現代
CSEP 報告的標準組合是 N + S + cL（M 視情況），而不是 L。

## 17.5 沒有解析分布就用模擬

N-test 有解析分布是因為 Poisson 的和還是 Poisson。S、M、cL 的檢定
統計量是**一大堆對數的加權和**，沒有任何標準分布長那樣。這時候唯一
的辦法是把分布**造出來**。

完整程序有五步，每一步都有一個容易踩的坑：

**第一步：把率場正規化成機率。** 依檢驗種類先做邊際加總與縮放得到
$\tilde\Lambda$，再除以總和得到各箱機率
$P_i = \tilde\Lambda_i/\sum_{i'}\tilde\Lambda_{i'}$。

**第二步：造累積分布。** $C_i = \sum_{i'\le i}P_{i'}$。

**第三步：抽 $U\sim\mathrm{Uniform}[0,1)$ 丟格。** 滿足
$C_{i-1}\le U<C_i$ 的那個 $i$ 就是這顆事件落的箱——這就是
{doc}`第 10 章 <10_point_process>`的反函數法，實作上是
`np.searchsorted`。重複 $N_{\rm obs}$ 次得到**一份合成目錄**；事件數
是**固定**的，這正是「條件在 $N_{\rm obs}$ 上」的操作定義。

**第四步：算分數，重複一萬次。** 每份合成目錄套 {eq}`eq:jpoll` 得到
一個分數，一萬份就構成檢定統計量的**參考分布**——「如果模型是對的，
分數會長什麼樣」的答案。

**第五步：讀分位數。** 記觀測分數為 $S_{\rm obs}$、第 $r$ 份模擬為
$S_r$，則

$$\gamma = \frac{1}{n_{\rm sim}}\sum_{r=1}^{n_{\rm sim}}
  \mathbf{1}\{S_r \le S_{\rm obs}\}$$

就是**分位數分數**。它在數值上等於單尾檢定的顯著性機率——$\gamma$
很小表示觀測分數低到模型自己幾乎模擬不出來，判定不一致。幾個實作上
的典型設定與它們的理由：

| 設定 | 典型值 | 理由 |
|---|---|---|
| 模擬次數 $n_{\rm sim}$ | 10,000 | 分位數的蒙地卡羅誤差 |
| 顯著水準 $\alpha$ | 0.05（或 0.01） | 歷史上並不統一 |
| 多重檢定校正 | $\alpha/2$ | 有效獨立檢驗數（17.8） |
| 零率箱底線 | 極小正值 | 避開 $\ln 0$ |

為什麼是一萬次而不是一千次？分位數分數是一個二項比例，在判定邊界
$\gamma\approx0.05$ 附近，一萬次模擬給出約 $0.002$ 的標準誤，一千次則
是 $0.007$——後者意味著「$\gamma=0.04$ 判不一致」這個結論本身有相當
機率是模擬雜訊（推導見附錄 E）。**檢驗結果的不確定性有兩層**：資料的
隨機性，以及你自己造分布時的隨機性；第二層可以用計算量買掉，就買掉。

下面用一個合成例子把 S-test 從頭跑一遍。模型是兩團高斯活動加上均勻
背景；「真實」的率場多了一團模型沒預期到的活動。

In [ ]:
rng = np.random.default_rng(17)

nx, ny = 30, 20                                  # 600 個空間格
gx = np.linspace(120.0, 122.0, nx)
gy = np.linspace(21.9, 25.3, ny)
GX, GY = np.meshgrid(gx, gy, indexing="ij")


def blob(x0, y0, s, amp):
    return amp * np.exp(-((GX - x0) ** 2 + (GY - y0) ** 2) / (2 * s ** 2))


bg = 0.05
lam_model = bg + blob(121.5, 24.0, 0.30, 1.0) + blob(120.7, 22.8, 0.25, 0.8)
lam_true = lam_model + blob(121.8, 22.6, 0.17, 2.4)   # 模型沒預期到的一團
lam_model, lam_true = lam_model.ravel(), lam_true.ravel()

N_obs = 60
p_true = lam_true / lam_true.sum()
omega = rng.multinomial(N_obs, p_true)           # 一份「觀測」

lam_s = lam_model * N_obs / lam_model.sum()      # S-test 的正規化率場
log_lam_s = np.log(lam_s)
const = lam_s.sum()


def jpoll(counts):
    return counts @ log_lam_s - const - gammaln(counts + 1).sum(axis=-1)


n_sim = 10_000
sims = rng.multinomial(N_obs, lam_s / lam_s.sum(), size=n_sim)
scores_sim = jpoll(sims)
score_obs = jpoll(omega)
gamma = float(np.mean(scores_sim <= score_obs))

fig = go.Figure(go.Histogram(x=scores_sim, nbinsx=60, marker_color=ACCENT,
                             opacity=0.8, name="模擬分數（一萬份合成目錄）"))
fig.add_vline(x=score_obs, line_color=QUAKE_COLOR, line_width=3,
              annotation_text=f"觀測分數 {score_obs:.1f}（γ={gamma:.4f}，"
                              f"n={n_sim:,}）")
fig.add_vline(x=float(np.quantile(scores_sim, 0.05)), line_dash="dash",
              line_color="#888", annotation_text="5% 分位")
apply_layout(fig,
             title=f"S-test：觀測分數落在模擬分布的第 {gamma * 100:.2f} 百分位",
             xaxis_title="空間聯合對數概似 jPOLL",
             yaxis_title="模擬次數", hovermode="x", showlegend=False)
fig

觀測分數落在模擬分布的左尾外，$\gamma$ 遠小於 0.05：**模型與觀測的
空間圖形不一致**。診斷訊息很明確——有一批地震落在模型認為率很低的
地方。這正是 S-test 該有的樣子：它不告訴你模型好不好，它告訴你
**模型在哪個面向與資料不合**。也請注意模擬分布本身相當窄：60 顆事件
落在 600 格裡，每顆貢獻的 $\ln\tilde\Lambda_j$ 變異有限；格子數與
事件數的比例一旦改變，這個分布的寬度、乃至整個檢驗的鑑別力都會跟著
變——這是第 18 章功效討論的伏筆。

### 誰主宰了分數

總分是六百格相加，但這六百項的貢獻極度不平均。把觀測分數與模擬分數的
**逐格差異**畫出來就能看見：

In [ ]:
cell_obs = omega * log_lam_s - lam_s - gammaln(omega + 1)
cell_exp = sims.mean(axis=0) * log_lam_s - lam_s - gammaln(sims + 1).mean(axis=0)
deficit = cell_exp - cell_obs                    # 每格「拖累」總分多少
order = np.argsort(-deficit)
share = np.cumsum(deficit[order]) / deficit[deficit > 0].sum()
top5, top20 = share[4] * 100, share[19] * 100

fig = make_subplots(rows=1, cols=2, column_widths=[0.58, 0.42],
                    subplot_titles=("逐格分數缺口的空間分布",
                                    "累積貢獻：少數格主宰總分"))
fig.add_trace(go.Heatmap(z=deficit.reshape(nx, ny).T, x=gx, y=gy,
                         colorscale=SEQUENTIAL, colorbar=dict(x=0.46,
                                                              title="缺口"),
                         hovertemplate="經度 %{x:.2f}<br>緯度 %{y:.2f}"
                                       "<br>缺口 %{z:.2f}<extra></extra>"),
              row=1, col=1)
hit = omega > 0
fig.add_trace(go.Scatter(x=GX.ravel()[hit], y=GY.ravel()[hit], mode="markers",
                         marker=dict(color=QUAKE_COLOR, size=5,
                                     line=dict(width=0)),
                         name="有事件的格", showlegend=False),
              row=1, col=1)
fig.add_trace(go.Scatter(x=np.arange(1, 41), y=share[:40] * 100, mode="lines",
                         line=dict(color=ACCENT, width=2.5), showlegend=False),
              row=1, col=2)
fig.add_hline(y=top20, line_dash="dash", line_color=QUAKE_COLOR,
              annotation_text=f"前 5 格佔 {top5:.0f}%，"
                              f"前 20 格佔 {top20:.0f}%", row=1, col=2)
fig.update_xaxes(title_text="經度", row=1, col=1)
fig.update_yaxes(title_text="緯度", row=1, col=1)
fig.update_xaxes(title_text="按缺口排序的格子名次", row=1, col=2)
fig.update_yaxes(title_text="累積佔正缺口的百分比", row=1, col=2)
apply_layout(fig, title="總分的來源極度不均：幾格決定了整個檢驗結果",
             hovermode="closest", showlegend=False)
fig

左圖是逐格的「分數缺口」（模擬期望減觀測），紅點是實際有事件的格，
深色的那幾格幾乎全部落在模型沒預期到的那一團裡。右圖把缺口由大到小
排序後累積：**六百格中的前 20 格（3%）就吃掉了六成的缺口**，前 5 格
佔近兩成（精確比例由圖中虛線標出）。

這個現象在真實實驗裡更極端。Bayona et al.（2022）的加州前瞻檢驗中，
2016 年 Hawthorne 群震（3 顆）與 2019 年 Ridgecrest 序列（8 顆）擠在
極少數格子裡，Poisson 對這幾格的懲罰幾乎主導了所有模型的成敗。

為什麼懲罰會這麼重？因為 $\omega\ln\Lambda$ 對 $\omega$ 是**線性**的：
一格出現 3 顆事件而模型給的 $\Lambda=10^{-3}$，這一格就貢獻
$3\ln(10^{-3})\approx-20.7$ 個對數單位，出現 1 顆只貢獻 $-6.9$。
**Poisson 認為「同一格出現三顆」是三次獨立的意外**，而在真實地震裡，
那往往是同一件事——一個序列——被數了三次。

診斷到這裡，兩條放寬 Poisson 的路線就都有了動機。17.6 節放寬**總數
的變異數**，17.7 節放寬**單格的計數方式**。

## 17.6 放寬 Poisson I：負二項

回到 17.4 埋的伏筆。Poisson 只有一個參數，平均與變異數被綁死成同一個
數；真實地震活動的年際變異遠大於此——一年有沒有一個大序列，總數可以
差好幾倍。這種**過度離散**（overdispersion）不是資料髒，而是叢集的
直接數學後果：附錄 B 會證明，一個分支比為 $n$ 的觸發過程，其計數的
變異數／平均比正好是 $1/(1-n)^2$，以典型的 $n\approx0.5$ 計就是
Poisson 的**四倍**，$n\to1$ 時發散。

負二項分布（negative binomial distribution, NBD）多一個參數，可以把
變異數與平均拆開：

$$p(\omega\mid\tau,\nu)
  = \frac{\Gamma(\tau+\omega)}{\Gamma(\tau)\,\omega!}\,
  \nu^{\tau}(1-\nu)^{\omega},
  \qquad \omega = 0,1,2,\dots$$

其動差為

$$\mu = \tau\frac{1-\nu}{\nu},
  \qquad \sigma^{2} = \tau\frac{1-\nu}{\nu^{2}}$$

（記號警告：這裡的 $\tau$ 與 $\nu$ 是**負二項的兩個參數**，只在本節
與附錄 D 使用；第 18 章的 Molchan 圖會用同樣兩個字母代表警報時空
比例與漏報率，是完全不同的量。）

兩式相除立刻得到一個關鍵性質：

$$\frac{\sigma^{2}}{\mu} = \frac{1}{\nu} > 1 , \qquad 0<\nu<1$$

**負二項永遠是過度離散的**，而且 $\nu\to1$ 時退化回 Poisson——剛好
是我們要的那個放寬方向。

### 實作訣竅：平均取自模型，變異數取自歷史

這是整節最重要的一句話。模型交出來的東西只有期望數，它**沒有交出
變異數**——網格化預報的格式裡根本沒有那一欄。變異數只能從別的地方來，
標準作法是**從歷史目錄的不重疊時段估計**，再把參數反解：

$$\begin{aligned}
\nu &= \frac{\mu}{\sigma^{2}}, \\
\tau &= \frac{\mu\,\nu}{1-\nu}
  = \frac{\mu^{2}}{\sigma^{2}-\mu}
\end{aligned}$$

（推導見附錄 D。）$\tau>0$ 要求 $\sigma^2>\mu$：若歷史估出來的變異數
反而小於模型平均（強力除叢過的目錄上有可能），NBD 無解，該退回 Poisson。

「不重疊時段」四個字不能省。要估「八年期的地震總數變異數」，就得把
歷史切成一段段**互不重疊的八年**，每段數一個總數，再算樣本變異數：
用滑動窗會嚴重低估變異數（相鄰窗共用大部分事件），用比預報期短的窗
則會低估叢集在長時間尺度上的累積效應。Bayona et al.（2022）用 ANSS
目錄 1932–2010 的**不重疊十年期**估出全加州的
$\sigma^2_C\approx314.21$；同一套邏輯後來被寫進 pyCSEP 的
`binomial_number_test()`。下圖把同樣的平均、不同的變異數放在同一軸上
比較。

In [ ]:
mu_nb = N_fore                                   # 與 N-test 圖同一個平均
sigma2_nb = 80.0                                 # 由歷史不重疊時段估得
nu_nb = mu_nb / sigma2_nb
tau_nb = mu_nb ** 2 / (sigma2_nb - mu_nb)
kk2 = np.arange(0, 61)
pmf_nb = stats.nbinom.pmf(kk2, tau_nb, nu_nb)
lo_nb, hi_nb = stats.nbinom.ppf([0.025, 0.975], tau_nb, nu_nb)

fig = go.Figure()
fig.add_trace(go.Bar(x=kk2, y=stats.poisson.pmf(kk2, mu_nb), opacity=0.65,
                     marker_color=ACCENT,
                     name=f"Poisson：μ={mu_nb:.0f}, σ²={mu_nb:.0f}"))
fig.add_trace(go.Bar(x=kk2, y=pmf_nb, opacity=0.65, marker_color=PALETTE[3],
                     name=f"負二項：μ={mu_nb:.0f}, σ²={sigma2_nb:.0f}"
                          f"（τ={tau_nb:.2f}, ν={nu_nb:.2f}）"))
fig.add_vline(x=obs_b, line_color=QUAKE_COLOR, line_width=3,
              annotation_text=f"觀測 B={obs_b}")
apply_layout(fig, title=f"同一個平均、不同的變異數："
                        f"Poisson 95% 區間 [{lo:.0f}, {hi:.0f}]，"
                        f"負二項 [{lo_nb:.0f}, {hi_nb:.0f}]",
             xaxis_title="預報期間的目標地震數", yaxis_title="機率",
             barmode="overlay", hovermode="x")
fig

圖中的變異數取 $\sigma^2=4\mu$，正是分支比 $n=0.5$ 所對應的離散度。
兩個分布的平均完全一樣，但負二項的右尾長得多。剛才被 Poisson N-test
判為「顯著低估」的觀測 B，在負二項 N-test 下**落在 95% 區間之內**——
同一份資料、同一個模型，換一個計數分布，結論就反轉了。

這件事該怎麼理解？它**不是**「負二項比較寬鬆所以比較好過」這種投機說
法。Poisson N-test 檢的是一個**複合假設**——「模型的率對」而且「計數是
Poisson」——被拒絕時你不知道是哪一半錯；負二項版本把第二個假設換成一
個由歷史校準過的、更接近真實的假設，拒絕訊息才乾淨地指向第一半。

代價當然有。$\sigma^2$ 是**估計出來的**，自己帶著不確定性，而檢驗
沒有把這層不確定性算進去；用歷史估變異數等於假設離散程度**平穩**，
這在剛發生完大地震的時期顯然不成立；而且 NBD 只修了**總數**的分布，
S-test 與 cL-test 的逐格 Poisson 假設原封不動——那要靠下一節。

## 17.7 放寬 Poisson II：二元似然

17.5 節的診斷指向一個很具體的病灶：$\omega\ln\Lambda$ 對 $\omega$ 是
線性的，一格三顆就罰三倍，而那三顆常常是同一個序列。既然問題出在
「數幾顆」，那就**別數**——只問這格**有沒有**地震。這就是 binary
（Bernoulli）似然的全部想法。從同一個 Poisson 假設出發，一格有事件
與沒事件的機率分別是

$$P(\omega = 0\mid\Lambda) = e^{-\Lambda},
  \qquad P(\omega \ge 1\mid\Lambda) = 1 - e^{-\Lambda}$$

定義指示變數 $X_{jk}=\mathbf{1}\{\omega_{jk}\ge1\}$（這格是不是
**active cell**），則單格分數 **BILL**（binary log-likelihood）是
一個 Bernoulli 對數概似：

$$\mathrm{BILL}(X,\Lambda)
  = X\ln\!\bigl(1-e^{-\Lambda}\bigr) + (1-X)\ln\!\bigl(e^{-\Lambda}\bigr)$$

逐箱相加得到 jBILL：

$$\mathrm{jBILL} = \sum_{j=1}^{l}\sum_{k=1}^{m}
  \Bigl[X_{jk}\ln\bigl(1-e^{-\Lambda_{jk}}\bigr)
  + \bigl(1-X_{jk}\bigr)\ln\bigl(e^{-\Lambda_{jk}}\bigr)\Bigr]$$

### 三個必須自己算一遍的性質

**性質一：$\omega=0$ 時 POLL 與 BILL 完全相同。** 代入即可，

$$\mathrm{POLL}(0,\Lambda) = -\Lambda - \ln 0! = -\Lambda
  = \ln e^{-\Lambda} = \mathrm{BILL}(0,\Lambda)$$

兩者**恆等**。這一點比看起來重要：典型 CSEP 設定裡九成九以上的箱是
空的，所以 jPOLL 與 jBILL 的絕大部分項是同一個數字。

**性質二：$\omega=1$ 且 $\Lambda\to0$ 時兩者近似。** 對小的 $\Lambda$，
$1-e^{-\Lambda}=\Lambda\bigl(1-\Lambda/2+O(\Lambda^{2})\bigr)$，取對數
並用 $\ln(1-u)=-u+O(u^2)$：

$$\begin{aligned}
\mathrm{BILL}(1,\Lambda) &= \ln\Lambda - \frac{\Lambda}{2} + O(\Lambda^{2}), \\
\mathrm{POLL}(1,\Lambda) &= \ln\Lambda - \Lambda, \\
\mathrm{POLL} - \mathrm{BILL} &= -\frac{\Lambda}{2} + O(\Lambda^{2})
  \;\xrightarrow[\Lambda\to0]{}\; 0
\end{aligned}$$

差異是 $O(\Lambda)$，而 CSEP 網格上的 $\Lambda$ 動輒是 $10^{-4}$ 量級，
所以**單事件格的兩種分數在數值上無法區分**。

**性質三：只有 $\omega\ge2$ 才有實質差異。** 關鍵在於 BILL **飽和**：
它只看 $X$，$\omega=2$ 或 $\omega=20$ 給的分數完全一樣，POLL 則繼續
隨 $\omega$ 線性下沉。兩者之差為

$$\begin{aligned}
\mathrm{POLL} - \mathrm{BILL}
  &= \bigl(-\Lambda + \omega\ln\Lambda - \ln\omega!\bigr)
     - \ln\bigl(1-e^{-\Lambda}\bigr) \\
  &= (\omega-1)\ln\Lambda - \ln\omega! - \frac{\Lambda}{2}
     + O(\Lambda^{2})
\end{aligned}$$

第一項 $(\omega-1)\ln\Lambda$ 是主角：$\Lambda\ll1$ 時 $\ln\Lambda$ 是
個大負數，所以**每多一顆事件，Poisson 就多罰 $|\ln\Lambda|$ 個對數
單位**。以 $\Lambda=10^{-3}$、$\omega=3$ 為例，兩者相差約
$2\times(-6.91)-\ln 6\approx-15.6$——單獨一格就能翻轉整個檢驗。完整
展開見附錄 C。

In [ ]:
lam_grid = np.logspace(-5, 0, 200)
fig = go.Figure()
for idx, w in enumerate([0, 1, 2, 3, 5]):
    poll = -lam_grid + w * np.log(lam_grid) - gammaln(w + 1)
    bill = np.where(w >= 1, np.log1p(-np.exp(-lam_grid)), -lam_grid)
    fig.add_trace(go.Scatter(x=lam_grid, y=poll - bill, mode="lines",
                             name=f"ω={w}",
                             line=dict(color=PALETTE[idx], width=2.5)))
diff3 = float((-1e-3 + 3 * np.log(1e-3) - gammaln(4))
              - np.log1p(-np.exp(-1e-3)))
fig.add_annotation(x=np.log10(1e-3), y=diff3, text=f"Λ=10⁻³, ω=3：{diff3:.1f}",
                   showarrow=True, arrowhead=2, ax=60, ay=-30)
fig.update_xaxes(type="log")
apply_layout(fig, title="POLL 減 BILL：只有多震格才有實質差異",
             xaxis_title="該格的預報期望數 Λ",
             yaxis_title="POLL − BILL（對數單位）", hovermode="x")
fig

$\omega=0$ 的線壓在零上（性質一），$\omega=1$ 的線在
$\Lambda\to0$ 時趨近零（性質二），$\omega\ge2$ 的線則往下發散
（性質三）——而且愈往左、也就是模型愈認為不可能的格子，差異愈大。
**二元似然把叢集的話語權從「多顆事件的格子」手上收回來。**

### binary S-test 的正規化

二元版的一致性檢驗要重新設計正規化。Poisson S-test 把率場縮放到
$\sum_j\tilde\Lambda_j=N_{\rm obs}$；二元版沒有「事件數」這個量，對應
物是 **active cell 數** $N_A=\sum_j X_j$，所以縮放條件改成「預期的
active cell 數等於觀測的 active cell 數」，即解出純量 $a$ 使

$$\sum_{j}\Bigl(1 - e^{-a\Lambda_{j}}\Bigr) = N_A$$

左式對 $a$ 嚴格單調遞增，二分法幾步就收斂，沒有數值上的麻煩。模擬時
同樣固定 active cell 數為 $N_A$。

### 適用邊界：一個誠實的反例

二元似然不是萬靈丹，而 Bayona et al.（2022）自己提供了最好的反例：
該研究 2011–2020 的加州前瞻期只有 40 顆 $M\ge4.95$ 目標地震，散在
七千多格裡，幾乎每顆各佔一格，$\omega\ge2$ 的格子屈指可數，結果
**Poisson 與 binary 的 cL-test 結論幾乎沒有差別**，差異只在有叢集的
S-test 才顯現。

由此得到一條可以直接用的判準：**二元似然只在「多顆事件會落進同一箱」
的設定下才有意義**——格子夠大、規模門檻夠低、或預報期夠長。台灣的
情境很符合：測試區小、序列叢集強烈（車籠埔、池上、花蓮外海），
$\omega\ge2$ 的格子不會罕見；反過來說，實驗設定裡幾乎沒有多震格時，
跑二元版只是多一組長得一樣的數字。

## 17.8 多重檢定

到目前為止，每個檢驗都是獨立地用 $\alpha=0.05$ 判定。但一個模型要跑
N、M、S、cL 四個檢驗，如果模型其實是對的，四個檢驗各有 5% 機率誤判，
**至少一個亮紅燈**的機率就不是 5%。在完全獨立的極端下，

$$P\bigl(\textstyle\bigcup_t A_t\bigr) = 1 - (1-\alpha)^{T}$$

其中 $A_t$ 是「第 $t$ 個檢驗誤判」的事件。$T=4$ 時這是 18.5%——一個
完美的模型有將近五分之一的機會被貼上「與資料不一致」的標籤，在同時
評估幾十個模型的實驗裡是災難。

標準的解方是 **Bonferroni 校正**：把單一檢驗的門檻降成
$\alpha_B=\alpha/T$，則整族檢驗的族群誤判率（family-wise error rate）
不超過 $\alpha$。證明只需要 Boole 不等式，不需要獨立性假設：
$P(\bigcup_t A_t)\le\sum_t P(A_t)=T\cdot(\alpha/T)=\alpha$。

### $T$ 該取多少：有效獨立檢驗數

這裡有個真正的地震學問題。Bonferroni 是保守的——檢驗之間如果高度
相關，除以 $T$ 會過度懲罰，讓檢驗力白白流失。而 N、M、S、cL 顯然
**不是**四個獨立的問題：cL-test 的統計量裡本來就包含了空間維度，
跟 S-test 共用大量資訊。

Bayona et al.（2022）用一個很漂亮的辦法量化了這件事：**拿一個模型
當資料產生器**（該研究用 HKJ），模擬 1000 份合成觀測，每份都跑完整
的檢驗組合，再算各檢驗**分位數分數之間的相關係數**。結果是 S-test
與 cL-test 的分位數分數**高度相關**（cL 含 S，合理），而兩者都與
N-test **幾乎獨立**（$R_{N,S}\approx0.01$、$R_{N,cL}\approx0.03$）。

換句話說，這一組檢驗其實只有**兩個真正獨立的資訊軸：數量，與形狀**。
於是有效獨立檢驗數取 $T=2$、$\alpha_B=0.05/2=0.025$，而不是 $T=4$；
N-test 的雙尾各取 $0.0125$，S 與 cL 的單尾各取 $0.025$。

這個結論有兩層價值。表面那層是「校正時該用有效檢驗數」；深一層是
**它揭示了四件套的真實結構**——17.4 節那張表看起來是四個獨立面向，
統計上其實是「率」與「形狀」兩軸。設計檢驗組合時該問的不是「跑得夠不
夠多」，而是「這些檢驗彼此獨立嗎」。

## 17.9 常見誤解與陷阱

**誤解一：「模型通過了所有一致性檢驗，所以它是好模型。」**
這是本章最需要打破的一句話。一致性檢驗是**否證工具**：沒有拒絕，
邏輯上只代表「在此網格、此樣本數、此顯著水準下，資料不足以拒絕該
模型」，完全沒有排除「檢驗根本沒有力氣拒絕任何東西」的可能——而這
在地震學裡是常態。Khawaja et al.（2023）示範過一個極端案例：一個
宣稱「地球上每個地方地震率都一樣」的均勻模型，在慣用的 $0.1^\circ$
全球網格上**通過了 S-test**。任何「本模型通過所有一致性檢驗」的宣稱
都必須附上該設定下的**統計功效**，否則是空話——功效是
{doc}`第 18 章 <18_testing_comparison>`的主題。

**誤解二：「檢驗不通過就該淘汰模型。」**
CSEP 自己的立場恰恰相反：**不因為某個檢驗失敗就正式 reject 模型**，
而是把分位數分數當作**診斷指標**。這不是鄉愿：一次拒絕可能來自模型
錯、分布假設錯（Poisson）、目錄問題（$M_c$ 變動、規模尺度不一致），
也可能就是那 5%。**檢驗是診斷，不是判決**——真正的判決要靠第 18 章的
模型間比較，加上跨多個時間窗的重複觀察。

**誤解三：「似然分數低就是模型爛。」**
17.5 節那張圖已經證明：少數幾格可以主宰整個分數，一個序列擠進三格
就能把一份原本合理的預報打成「不一致」。看到低分時的第一件事是**打開
逐格分解**，看看分數是均勻地差（模型的系統性問題）還是被幾格拉垮的
（往往是 Poisson 假設與叢集的衝突，該換成二元或負二項版本重測）。

**誤解四：「解析度愈高，檢驗愈嚴格。」**
直覺說格子愈細、要求愈精確，實際上恰好相反。格子細到每顆地震各佔
一格時 $\omega_{jk}\in\{0,1\}$，分數幾乎只由「有幾格中了一顆」決定，
「中在哪裡」的資訊被解析度稀釋殆盡；細網格同時把 S-test 變成了叢集
偵測器——這不是原本要檢的東西。網格是研究者自己選的變數，**檢驗
設計本身就是實驗設計**，第 18 章談功效時會有量化證據。

**誤解五：「同時跑很多檢驗比較保險。」**
見 17.8 節。跑愈多檢驗，至少一個假陽性的機率愈高；不做校正就報「本
模型只有一項不通過」，這句話幾乎沒有資訊量。

**誤解六：「jPOLL 可以跨實驗比較。」**
不行。17.3 節證明過 jPOLL 含有 $N\ln\Delta V$ 這個只跟解析度有關的
項，還隨格子總數與事件數變動。要比就比**每顆地震的平均分數**
（Poisson）或**每個 active cell 的平均分數**（二元）。

**陷阱一：$-\infty$。** 任何含對數的分數，只要有一顆地震落在預報率
為零的箱，總分立刻是 $-\infty$，整份預報報銷——這在低活動區極常見。
實務上要為零率箱設一個 water level 底線率，而這個底線是**研究者的
選擇**、會影響結果，必須事前釘死並寫進預報規格（回到 17.1 的零自由度）。

**陷阱二：定義不對齊。** 規模尺度（$M_L$ 與 $M_w$ 不能直接混用）、
深度範圍、是否除叢、目錄版本——任何一項不對齊，後面的統計全是空談。
Bayona et al.（2023）把全球模型 GEAR1 下放到區域測試時逐項檢查並換算
（用 $b=1$ 外推規模門檻、義大利因尺度差異把率除以 1.602、實證檢查
深部事件貢獻可忽略），是標準示範。

**陷阱三：模擬的隨機性沒有被報告。** $\gamma=0.048$ 與 $\gamma=0.052$
在一萬次模擬下是同一件事，卻會被寫成「不通過」與「通過」兩種結論；報
$\gamma$ 時該一併報 $n_{\rm sim}$。

## 17.10 研究前沿與未解問題

### 目錄式檢驗

17.2 節提過的第二種表示法，正在把整個檢驗生態往外推。目錄式預報交出
一萬份模擬目錄，於是每一個一致性檢驗都有了**不需要 Poisson 假設**的
類比物（Savran et al. 2020）：N-test 直接比對模擬目錄的事件數經驗
分布（自然帶著過度離散，不必外掛負二項）；M-test 比對逐箱的增量規模
分布差；S-test 比對目標事件率的**幾何平均**；另有一個仿連續點過程
概似的 pseudolikelihood test。判準改用兩個經驗分位數 $\delta_1$
（模擬目錄中事件數 $\ge$ 觀測的比例）與 $\delta_2$（$\le$ 觀測）。

未解的部分不少：目錄式檢驗的統計性質（功效、對模擬份數的敏感度、
檢驗之間的相關結構）遠不如網格化版本清楚；而「模擬一萬份目錄」對計算
資源與模型可執行性的要求，也把一部分模型排除在外。

### 評分函數的正當性

更根本的一條線是：**我們憑什麼用對數概似當分數？** 一般化的問法要
借用預報學的 **scoring rule** 框架——一個把「預報 $P$」與「資料 $D$」
映到分數的函數 $S(P,D)$。一個評分函數是 **proper** 的，若「當被評估
的預報等於真實分布時，期望分數最佳」。觀測聯合對數概似是 proper
score，這正是它有資格當共同貨幣的理由；用 improper score 排名機率式
地震預報則會產生系統性偏誤（Serafini et al. 2022）。但 log score 有
一個結構性弱點，就是上一節的 $-\infty$：它對零機率箱給無限懲罰。二元
事件的 **Brier score**（基於預報機率與觀測之間的平方差）同樣 proper，
卻**永遠有限**，Graham et al.（2024）因此建議 **Brier score 與 log
score 併看**。評分函數的選擇會如何改變模型排名是目前活躍的問題，
{doc}`第 18 章 <18_testing_comparison>`的 18.6 節會正面處理。

### CSEP 第二階段：把裁判的尺攤在陽光下

CSEP 第一階段的檢驗程式封在各地的 testing center 裡，外人無法檢視——
一個以可重現性為存在理由的計畫，自己的評分程式不可重現。第二階段的
解方是 **pyCSEP**（Savran et al. 2022；Graham et al. 2024）：把目錄
存取、預報表示、統計檢驗、視覺化四個模組重寫成開源 Python 套件，並
附上**可重現性套件**（程式碼、資料、凍結的執行環境、指定版本，一道
指令重跑全部圖表）。它的意義超過方便——**可重現性不是附錄，是方法論
的一部分**；它同時讓「floating experiment」成為可能：研究者不必等
中心化的實驗開跑，可以自己宣告規格、自己前瞻評估、公開存證。

剩下的難題是老問題：前瞻資料累積極慢。台灣測試區小、$M\ge5$ 的目標事
件少，正落在最不利的區間；可能的出路是空間換時間（多區域並行、全球實
驗），以及用資料驅動的多解析度網格提高單位資料的檢驗力——都通向下一章。

## 17.11 附錄：本章推導細節

### A. 網格化分數與連續分數的對應

設箱體積 $\Delta V=\Delta t\,\Delta A\,\Delta m$ 小到箱內 $\lambda^*$
可視為常數，且每箱至多一顆事件。對空箱 $\mathrm{POLL}=-\Lambda_{jk}$；
對含一顆事件的箱（設該事件為第 $i$ 顆），

$$\begin{aligned}
\mathrm{POLL} &= -\Lambda_{jk} + \ln\Lambda_{jk} - \ln 1! \\
  &= -\Lambda_{jk} + \ln\bigl(\lambda^*(t_i,x_i,y_i,m_i)\,\Delta V\bigr) \\
  &= -\Lambda_{jk} + \ln\lambda^*(t_i,x_i,y_i,m_i) + \ln\Delta V
\end{aligned}$$

對所有箱求和：$-\sum_{j,k}\Lambda_{jk}$ 是 Riemann 和，$\Delta V\to0$
時收斂到 $\int\lambda^*$；含事件的箱共 $N$ 個，各貢獻一個
$\ln\lambda^*$ 與一個 $\ln\Delta V$。故

$$\mathrm{jPOLL} = \left[\sum_{i=1}^{N}\ln\lambda^*_i
  - \int\lambda^*\right] + N\ln\Delta V + o(1)$$

方括號裡正是 {eq}`eq:pp-loglik`。常數 $N\ln\Delta V$ 只依賴解析度與
事件數、不依賴模型，因此在**同一個網格上比較兩個模型**時消掉；但它
使得 jPOLL 的絕對值與跨網格的比較都沒有意義。

### B. 為什麼叢集必然造成過度離散

把一段時間內的事件數 $N$ 分解成「背景事件各自帶起一整叢」。設背景
事件數 $B\sim\mathrm{Poisson}(\mu_B)$，第 $i$ 個背景事件連同其所有
後代共 $G_i$ 顆（獨立同分布），則 $N=\sum_{i=1}^{B}G_i$ 是**複合
Poisson**，標準的複合分布動差公式（並用 Poisson 的
$\mathrm{Var}(B)=\mathbb{E}[B]$）給出 $\mathbb{E}[N]=\mu_B\mathbb{E}[G]$、
$\mathrm{Var}(N)=\mu_B\mathbb{E}[G^{2}]$，於是

$$\frac{\mathrm{Var}(N)}{\mathbb{E}[N]}
  = \frac{\mathbb{E}[G^{2}]}{\mathbb{E}[G]} \;\ge\; \mathbb{E}[G] \;\ge\; 1$$

只要叢集大小 $G$ 不是恆等於 1，這個比值就大於 1——**過度離散是叢集的
數學必然**。

進一步，若每顆事件的直接後代數服從平均為 $n$（分支比，第 13 章）的
Poisson 分布，則 $G$ 是 Galton–Watson 過程的總後代數，其動差為
$\mathbb{E}[G]=1/(1-n)$、$\mathrm{Var}(G)=n/(1-n)^{3}$（次臨界
$n<1$）。代入：

$$\begin{aligned}
\frac{\mathbb{E}[G^{2}]}{\mathbb{E}[G]}
  &= (1-n)\left[\frac{n}{(1-n)^{3}} + \frac{1}{(1-n)^{2}}\right] \\
  &= \frac{n}{(1-n)^{2}} + \frac{1}{1-n}
  = \frac{n + (1-n)}{(1-n)^{2}}
  = \frac{1}{(1-n)^{2}}
\end{aligned}$$

乾淨得出奇：$n=0.5$ 時變異數是 Poisson 的 4 倍，$n=0.8$ 時是 25 倍。
這條式子同時解釋了為什麼 N-test 需要負二項，以及為什麼**除叢過的
目錄**（人為把 $n$ 壓到 0）反而比較接近 Poisson。

### C. POLL 與 BILL 的完整展開

對 $\omega\ge1$（此時 $X=1$），

$$\begin{aligned}
\mathrm{POLL} - \mathrm{BILL}
  &= \bigl(-\Lambda + \omega\ln\Lambda - \ln\omega!\bigr)
     - \ln\bigl(1-e^{-\Lambda}\bigr) \\
  &= -\Lambda + \omega\ln\Lambda - \ln\omega!
     - \ln\Lambda - \ln\frac{1-e^{-\Lambda}}{\Lambda} \\
  &= (\omega-1)\ln\Lambda - \ln\omega! - \Lambda
     - \ln\left(1 - \frac{\Lambda}{2} + \frac{\Lambda^{2}}{6} - \cdots\right) \\
  &= (\omega-1)\ln\Lambda - \ln\omega! - \frac{\Lambda}{2}
     + O(\Lambda^{2})
\end{aligned}$$

第三行用了
$\bigl(1-e^{-\Lambda}\bigr)/\Lambda = 1-\Lambda/2+\Lambda^{2}/6-\cdots$，
第四行用了 $-\ln(1-u)=u+O(u^{2})$ 並與 $-\Lambda$ 合併。逐項讀：
$\omega=1$ 時第一、二項皆為零，只剩 $-\Lambda/2+O(\Lambda^2)$，得性質
二；$\omega\ge2$ 時第一項以 $|\ln\Lambda|$ 為單位線性放大，$\Lambda$
愈小差異愈大，得性質三。$\omega=0$ 不能用這個展開（$X=0$），直接
代入即得性質一。

### D. 負二項參數的反解

由 $\mu=\tau(1-\nu)/\nu$ 與 $\sigma^{2}=\tau(1-\nu)/\nu^{2}$ 相除得
$\sigma^{2}/\mu = 1/\nu$，即 $\nu = \mu/\sigma^{2}$；代回第一式：

$$\tau = \frac{\mu\,\nu}{1-\nu}
  = \frac{\mu\cdot\dfrac{\mu}{\sigma^{2}}}{1 - \dfrac{\mu}{\sigma^{2}}}
  = \frac{\mu^{2}}{\sigma^{2}-\mu}$$

定義域條件是 $\sigma^{2}>\mu$（否則 $\tau\le0$ 無意義）與 $0<\nu<1$
（由同一條件自動滿足）。$\sigma^{2}\to\mu^{+}$ 時 $\tau\to\infty$、
$\nu\to1$，負二項退化為 Poisson——這是把 Poisson 視為負二項極限的正式
說法。

### E. 分位數分數的蒙地卡羅誤差

令 $\gamma$ 為分位數分數的真值、$\hat\gamma$ 為 $n_{\rm sim}$ 次模擬的
估計。每次模擬的分數是否 $\le$ 觀測分數是一個 Bernoulli 試驗，故
$n_{\rm sim}\hat\gamma\sim\mathrm{Binomial}(n_{\rm sim},\gamma)$、
$\mathrm{SE}(\hat\gamma)=\sqrt{\gamma(1-\gamma)/n_{\rm sim}}$，在
$\gamma=0.05$ 下 $n_{\rm sim}=10^{4}$ 給 $0.0022$、$10^{3}$ 給
$0.0069$。若想報一個保守的判定，可用二項信賴區間的上界而非點估計。

---

回頭看這一章走過的路：從一條連續的對數概似出發，經過網格化、Poisson
假設、四個刻意殘缺的檢驗、一套模擬程序，再把 Poisson 假設從兩個方向
鬆開。整條路上最重要的訊息只有一句——**每一個檢驗都是刻意殘缺的，
而它的價值正來自那個殘缺**。N-test 看不見空間、S-test 看不見率、
二元似然看不見顆數，這些不是妥協而是設計：只有把面向拆開，紅燈才
攜帶診斷資訊。

也因此，本章的所有工具**都不能回答「哪個模型比較好」**。兩個模型可以
同時通過全部檢驗，也可以同時失敗；更尷尬的是，一個毫無資訊量的均勻
模型可能安然通過，而一個相當不錯的模型可能被一個群震序列打成不一致。
要排名，需要一個共同的基準、一個把分數差轉成「每顆地震賺多少」的量、
一個誠實面對「檢驗有沒有力氣」的功效分析。

{doc}`第 18 章 <18_testing_comparison>`要做的正是這件事——把視角從
「模型 vs. 資料」轉成「模型 vs. 模型」，從資訊增益 IGPE 與 T-test
開始，一路走到 Molchan 圖與面積技能分數，最後回頭質問這整套檢驗體系
自己的統計功效。屆時你會發現，本章那句「沒被拒絕不等於模型好」不只是
一句提醒，而是一個可以量化、而且量化結果相當難堪的命題。